# 🚕 NYC Taxi Lakehouse

## Silver Layer - Data Cleansing & Standardization

The purpose of the Silver layer is to transform raw Bronze data into a clean, consistent, and analytics-ready dataset.

Unlike the Bronze layer, this stage applies data quality checks, derives useful attributes, and prepares the data for downstream business reporting.

### Objectives

- Read data from the Bronze layer
- Profile the dataset
- Identify data quality issues
- Apply business rules
- Create derived columns
- Store the cleaned dataset as a Delta table

In [0]:
from pyspark.sql import functions as F

## Read Bronze Table

We'll begin by loading the raw Delta table created in the Bronze layer.

All transformations in this notebook will use the Bronze table as the source.

In [0]:
bronze_df = spark.table("taxi.bronze.yellow_taxi")

In [0]:
display(bronze_df.limit(10))

## Data Profiling

In [0]:
#We'll first review the size of the dataset and inspect the schema.

print(f"Total Rows    : {bronze_df.count()}")
print(f"Total Columns : {len(bronze_df.columns)}")

In [0]:
bronze_df.printSchema()

## Missing Values

Understanding missing values helps determine whether records should be removed, imputed, or left unchanged.

For the Silver layer, we'll identify which columns contain null values before defining any cleaning rules.

In [0]:
display(
    bronze_df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in bronze_df.columns
    ])
)

## Duplicate Records

Duplicate records can distort reporting and KPI calculations.

Let's determine whether the dataset contains any exact duplicate rows.

In [0]:
total_rows = bronze_df.count()

distinct_rows = bronze_df.distinct().count()

print(f"Total Rows     : {total_rows:,}")
print(f"Distinct Rows  : {distinct_rows:,}")
print(f"Duplicates     : {total_rows - distinct_rows:,}")

## Pickup Date Range

Reviewing the pickup date range confirms the reporting period covered by the dataset.

In [0]:
display(
    bronze_df.select(
        F.min("tpep_pickup_datetime").alias("Minimum Pickup"),
        F.max("tpep_pickup_datetime").alias("Maximum Pickup")
    )
)

## Fare Amount Analysis

Negative fares generally indicate invalid records or adjustment transactions.

We'll inspect whether any exist before defining our cleaning rules.

In [0]:
bronze_df.filter(
    F.col("fare_amount") < 0
).count()

In [0]:
display(
    bronze_df.filter(
        F.col("fare_amount") < 0
    )
)

## Trip Distance Analysis

Trip distance should normally be greater than zero.

We'll investigate records with negative or zero distances.

In [0]:
print(
    "Negative Distance:",
    bronze_df.filter(
        F.col("trip_distance") < 0
    ).count()
)

print(
    "Zero Distance:",
    bronze_df.filter(
        F.col("trip_distance") == 0
    ).count()
)

## Timestamp Validation

A trip cannot end before it begins.

Let's verify whether any records violate this rule.

In [0]:
invalid_time = bronze_df.filter(
    F.col("tpep_dropoff_datetime") <
    F.col("tpep_pickup_datetime")
)

print(
    f"Invalid Trips : {invalid_time.count():,}"
)

## Passenger Count Distribution

Passenger count is an important business attribute.

Reviewing its distribution helps identify unusual or unexpected values.

In [0]:
display(
    bronze_df.groupBy("passenger_count")
             .count()
             .orderBy("passenger_count")
)

## Payment Type Distribution

Understanding payment methods provides insight into customer behavior and helps validate categorical values.

In [0]:
display(
    bronze_df.groupBy("payment_type")
             .count()
             .orderBy("payment_type")
)

#Ratecode Distribution

In [0]:
display(
    bronze_df.groupBy("RatecodeID")
             .count()
             .orderBy("RatecodeID")
)

#Vendor Distribution

In [0]:
display(
    bronze_df.groupBy("VendorID")
             .count()
             .orderBy("VendorID")
)